# Essential Commands for the Capstone — R Demo

Dataset: **Campus Wellness Survey** (synthetic)

## Load the data

In [ ]:
df <- read.csv("campus_wellness.csv")
head(df)

## Explore structure

In [ ]:
dim(df)

In [ ]:
str(df)

## Data wrangling basics

### Selecting columns

In [ ]:
# Single column
df$sleep_hrs

In [ ]:
# Multiple columns by name
df[, c("sleep_hrs", "coffee_cups", "stress_score")]

### Selecting / filtering rows

In [ ]:
# By position — first 5 rows
df[1:5, ]

In [ ]:
# By condition — only students who do yoga
df[df$yoga == 1, ]

In [ ]:
# Combine: yoga practitioners under 22
subset(df, yoga == 1 & age < 22)

### Column-level aggregates

In [ ]:
# Sum, mean, median of one column
sum(df$sleep_hrs)
mean(df$sleep_hrs)
median(df$sleep_hrs)

In [ ]:
# Across several columns at once
quant <- c("sleep_hrs", "coffee_cups", "exercise_hrs")
sapply(df[quant], function(x) c(
  sum    = sum(x, na.rm = TRUE),
  mean   = mean(x, na.rm = TRUE),
  median = median(x, na.rm = TRUE)
))

## Missing values

In [ ]:
# Missing count per column
colSums(is.na(df))

In [ ]:
# Rows with at least one missing value
sum(!complete.cases(df))

## Five-point summary (quantitative variables)

In [ ]:
summary(df[, c("sleep_hrs", "coffee_cups", "exercise_hrs", "stress_score", "age", "study_hrs")])

## Counts & proportions (categorical variables)

In [ ]:
# Counts
table(df$sex)
table(df$yoga)
table(df$exam_pass)
table(df$semester)

In [ ]:
# Proportions
prop.table(table(df$sex))
prop.table(table(df$yoga))
prop.table(table(df$exam_pass))

## Univariate visualizations

In [ ]:
library(ggplot2)

ggplot(df, aes(x = sleep_hrs)) +
  geom_histogram(binwidth = 0.5, fill = "grey70", color = "black") +
  theme_minimal() + ggtitle("Sleep Hours")

In [ ]:
ggplot(df, aes(x = stress_score)) +
  geom_histogram(binwidth = 3, fill = "grey70", color = "black") +
  theme_minimal() + ggtitle("Stress Score")

In [ ]:
ggplot(df, aes(x = exercise_hrs)) +
  geom_histogram(binwidth = 0.5, fill = "grey70", color = "black") +
  theme_minimal() + ggtitle("Exercise Hours")

## Bar plot (categorical variable)

In [ ]:
ggplot(df, aes(x = factor(semester))) +
  geom_bar(fill = "grey70", color = "black") +
  theme_minimal() +
  labs(x = "Semester", y = "Count", title = "Students by Semester")

## Table 1 — stratified by sex

In [ ]:
# install.packages("tableone")
library(tableone)

vars <- c("sleep_hrs","coffee_cups","exercise_hrs","stress_score",
          "yoga","exam_pass","age","study_hrs")
tab1 <- CreateTableOne(vars = vars, strata = "sex",
                       data = df, factorVars = c("yoga","exam_pass"), test = TRUE)
print(tab1, showAllLevels = TRUE)

## Welch's t-test — stress_score by sex

In [ ]:
t.test(stress_score ~ sex, data = df)

In [ ]:
ggplot(df, aes(x = factor(sex), y = stress_score)) +
  geom_boxplot() + theme_minimal() +
  labs(x = "Sex", y = "Stress Score", title = "Stress Score by Sex")

## Correlation — stress_score vs sleep_hrs

In [ ]:
cor.test(df$stress_score, df$sleep_hrs)

In [ ]:
ggplot(df, aes(x = sleep_hrs, y = stress_score)) +
  geom_point(alpha = 0.3) + theme_minimal() +
  labs(x = "Sleep Hours", y = "Stress Score", title = "Stress Score vs Sleep Hours")

## Multiple linear regression — stress_score ~ sex + sleep_hrs + exercise_hrs + yoga

In [ ]:
model_lm <- lm(stress_score ~ sex + sleep_hrs + exercise_hrs + yoga, data = df)
summary(model_lm)

## Logistic regression — exam_pass ~ sex + sleep_hrs + yoga + study_hrs

In [ ]:
model_glm <- glm(exam_pass ~ sex + sleep_hrs + yoga + study_hrs,
                   data = df, family = binomial)
summary(model_glm)

In [ ]:
# Odds ratios + confidence intervals
exp(coef(model_glm))
exp(confint(model_glm))

## Stratified correlation — exercise_hrs vs age, by sex

In [ ]:
# Males
with(subset(df, sex == 1), cor.test(exercise_hrs, age))

In [ ]:
# Females
with(subset(df, sex == 2), cor.test(exercise_hrs, age))